# Model Training

## Feature Selection

In [3]:
# Step 6.1: Load Dataset

import pandas as pd

# Step 6.1: Load Feature-Engineered Dataset

train = pd.read_csv("../data/processed/train_features.csv")

print(train.shape)
print(train.columns.tolist())

(1017209, 29)
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'IsStateHoliday', 'Lag_1_Sales', 'Lag_7_Sales', 'Lag_14_Sales', 'Rolling_7_Sales', 'Rolling_14_Sales', 'Rolling_30_Sales']


In [4]:
# Step 6.1: Check Missing Values

print(train.isnull().sum())

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
Year                              0
Month                             0
Day                               0
WeekOfYear                        0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
IsStateHoliday                    0
Lag_1_Sales                    1115
Lag_7_Sales                    7805
Lag_14_Sales                  15610
Rolling_7_Sales                7805
Rolling_14_Sales            

In [5]:
# Step 6.1: Define ML Features

features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "Promo2",
    "IsStateHoliday",
    "Lag_1_Sales",
    "Lag_7_Sales",
    "Lag_14_Sales",
    "Rolling_7_Sales",
    "Rolling_14_Sales",
    "Rolling_30_Sales"
]

target = "Sales"

print("Number of Features:", len(features))
print("Target:", target)

Number of Features: 20
Target: Sales


In [6]:
# Step 6.2: Create X and y

X = train[features]
y = train[target]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (1017209, 20)
y Shape: (1017209,)


In [7]:
# Step 6.3: Check Missing Values in ML Data

print("Rows before removing missing values:", len(X))

missing_rows = X.isnull().any(axis=1).sum()

print("Rows with missing values:", missing_rows)

Rows before removing missing values: 1017209
Rows with missing values: 36002


In [8]:
# Step 6.3: Remove Rows with Missing ML Features

valid_rows = X.notnull().all(axis=1)

X = X[valid_rows]
y = y[valid_rows]

print("X Shape after cleaning:", X.shape)
print("y Shape after cleaning:", y.shape)

X Shape after cleaning: (981207, 20)
y Shape after cleaning: (981207,)


In [9]:
# Step 6.4: Convert Date

train["Date"] = pd.to_datetime(train["Date"])

print(train["Date"].min())
print(train["Date"].max())

2013-01-01 00:00:00
2015-07-31 00:00:00


In [10]:
# Step 6.5: Time-Based Train-Validation Split

split_date = train["Date"].quantile(0.8)

train_data = train[train["Date"] <= split_date]
valid_data = train[train["Date"] > split_date]

print("Split Date:", split_date)
print("Training Data:", train_data.shape)
print("Validation Data:", valid_data.shape)

Split Date: 2015-01-30 00:00:00
Training Data: (814279, 29)
Validation Data: (202930, 29)


In [11]:
# Step 6.6: Create X and y for Training and Validation

X_train = train_data[features]
y_train = train_data["Sales"]

X_valid = valid_data[features]
y_valid = valid_data["Sales"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

X_train: (814279, 20)
y_train: (814279,)
X_valid: (202930, 20)
y_valid: (202930,)


In [12]:
# Step 6.7: Check Feature Data Types

print(X_train.dtypes)

Store                    int64
DayOfWeek                int64
Open                     int64
Promo                    int64
SchoolHoliday            int64
Year                     int64
Month                    int64
Day                      int64
WeekOfYear               int64
StoreType                  str
Assortment                 str
CompetitionDistance    float64
Promo2                   int64
IsStateHoliday           int64
Lag_1_Sales            float64
Lag_7_Sales            float64
Lag_14_Sales           float64
Rolling_7_Sales        float64
Rolling_14_Sales       float64
Rolling_30_Sales       float64
dtype: object


In [13]:
# Step 6.8: Check Categorical Values

print("StoreType:", X_train["StoreType"].unique())
print("Assortment:", X_train["Assortment"].unique())

StoreType: <StringArray>
['c', 'a', 'd', 'b']
Length: 4, dtype: str
Assortment: <StringArray>
['a', 'c', 'b']
Length: 3, dtype: str


In [14]:
# Step 6.9: Encode Categorical Features

X_train = pd.get_dummies(X_train, columns=["StoreType", "Assortment"])
X_valid = pd.get_dummies(X_valid, columns=["StoreType", "Assortment"])

X_train, X_valid = X_train.align(X_valid, join="left", axis=1, fill_value=False)

print("X_train Shape:", X_train.shape)
print("X_valid Shape:", X_valid.shape)

X_train Shape: (814279, 25)
X_valid Shape: (202930, 25)


In [16]:
# Step 6.10: Remove Missing Values from Train and Validation

train_data = train_data.dropna(subset=features)
valid_data = valid_data.dropna(subset=features)

X_train = train_data[features]
y_train = train_data["Sales"]

X_valid = valid_data[features]
y_valid = valid_data["Sales"]

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

X_train: (778823, 20)
X_valid: (202384, 20)


In [17]:
# Step 6.11: Encode Categorical Features

X_train = pd.get_dummies(X_train, columns=["StoreType", "Assortment"])
X_valid = pd.get_dummies(X_valid, columns=["StoreType", "Assortment"])

X_train, X_valid = X_train.align(
    X_valid,
    join="left",
    axis=1,
    fill_value=False
)

print("X_train Shape:", X_train.shape)
print("X_valid Shape:", X_valid.shape)

X_train Shape: (778823, 25)
X_valid Shape: (202384, 25)


In [18]:
# Step 6.11: Check Missing Values

print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_valid:", X_valid.isnull().sum().sum())

Missing values in X_train: 0
Missing values in X_valid: 0


In [19]:
# Step 6.12: Train Linear Regression Model

from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

print("Linear Regression Model Trained Successfully")

Linear Regression Model Trained Successfully


In [20]:
# Step 6.13: Make Predictions

y_pred = model.predict(X_valid)

print("Predictions Created Successfully")
print("Number of Predictions:", len(y_pred))

Predictions Created Successfully
Number of Predictions: 202384


In [21]:
# Step 6.14: Evaluate Linear Regression Model

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_valid, y_pred)
rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
r2 = r2_score(y_valid, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 970.0509113687947
RMSE: 1403.2728673519598
R²: 0.8682392569274797


In [22]:
# Step 6.15: Train Random Forest Model

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest Model Trained Successfully")

Random Forest Model Trained Successfully


In [23]:
# Step 6.16: Make Random Forest Predictions

rf_pred = rf_model.predict(X_valid)

print("Random Forest Predictions Created Successfully")
print("Number of Predictions:", len(rf_pred))

Random Forest Predictions Created Successfully
Number of Predictions: 202384


In [24]:
# Step 6.17: Evaluate Random Forest Model

rf_mae = mean_absolute_error(y_valid, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_valid, rf_pred))
rf_r2 = r2_score(y_valid, rf_pred)

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)
print("Random Forest R²:", rf_r2)

Random Forest MAE: 498.66548852676095
Random Forest RMSE: 808.7457957016045
Random Forest R²: 0.9562351089979048


In [25]:
# Step 6.18: Train Gradient Boosting Model

from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    random_state=42
)

gb_model.fit(X_train, y_train)

print("Gradient Boosting Model Trained Successfully")

Gradient Boosting Model Trained Successfully


In [26]:
# Step 6.19: Make Gradient Boosting Predictions

gb_pred = gb_model.predict(X_valid)

print("Gradient Boosting Predictions Created Successfully")
print("Number of Predictions:", len(gb_pred))

Gradient Boosting Predictions Created Successfully
Number of Predictions: 202384


In [27]:
# Step 6.20: Evaluate Gradient Boosting Model

gb_mae = mean_absolute_error(y_valid, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_valid, gb_pred))
gb_r2 = r2_score(y_valid, gb_pred)

print("Gradient Boosting MAE:", gb_mae)
print("Gradient Boosting RMSE:", gb_rmse)
print("Gradient Boosting R²:", gb_r2)

Gradient Boosting MAE: 644.5356989847922
Gradient Boosting RMSE: 982.6460106035217
Gradient Boosting R²: 0.9353905610374638


In [28]:
# Step 6.21: Compare All Baseline Models

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        mae,
        rf_mae,
        gb_mae
    ],
    "RMSE": [
        rmse,
        rf_rmse,
        gb_rmse
    ],
    "R2": [
        r2,
        rf_r2,
        gb_r2
    ]
})

print(results)

               Model         MAE         RMSE        R2
0  Linear Regression  970.050911  1403.272867  0.868239
1      Random Forest  498.665489   808.745796  0.956235
2  Gradient Boosting  644.535699   982.646011  0.935391


In [29]:
# Step 6.22: Select Baseline Model

best_model_name = results.loc[results["MAE"].idxmin(), "Model"]

print("Baseline Model:", best_model_name)

Baseline Model: Random Forest


In [30]:
# Step 6.23: Final Verification

print("Models Evaluated:", len(results))
print("Baseline Model:", best_model_name)
print()
print(results)

Models Evaluated: 3
Baseline Model: Random Forest

               Model         MAE         RMSE        R2
0  Linear Regression  970.050911  1403.272867  0.868239
1      Random Forest  498.665489   808.745796  0.956235
2  Gradient Boosting  644.535699   982.646011  0.935391
